# Điều tra: "doanh thu tháng 12/2011 giảm 58%, vì sao?"

Sáng thứ Hai, Giám đốc kinh doanh gửi bạn ảnh chụp dashboard: doanh thu tháng 11/2011 là 1.51 triệu GBP,
tháng 12/2011 mới 0.64 triệu GBP. Kèm câu hỏi: **"Doanh thu tháng 12 giảm mạnh so với tháng 11, vì sao? Tôi
cần câu trả lời trước buổi họp HĐQT thứ Năm."**

Ở bài Intermediate 02 bạn đã thấy tháng 12/2011 chỉ có 8 ngày dữ liệu. Bài này đi hết vòng điều tra mà một
DA phải làm trước khi trả lời lãnh đạo: (1) xác nhận con số có so được không (độ phủ ngày, giờ cuối cùng
có dữ liệu); (2) so sánh công bằng (cùng số ngày với tháng trước và **cùng kỳ năm trước**, để trừ mùa vụ);
(3) kiểm con số đã so có bị một vài giao dịch bất thường quyết định không; (4) tách thay đổi theo quốc gia,
khách mới/cũ và sản phẩm; (5) viết `MEMO.md` một trang: headline, 3 chart, kết luận, việc cần làm.

Dữ liệu: bộ `online-retail-ii`. Sheet **`Year 2010-2011`** là dữ liệu điều tra (01/12/2010 tới 09/12/2011);
sheet **`Year 2009-2010`** chỉ dùng làm **lịch sử** để biết khách nào đã từng mua (hai sheet trùng nhau
01-09/12/2010, nên đừng cộng doanh thu hai sheet với nhau).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. `clean` đã viết sẵn (quy tắc như các bài trước); ba hàm còn lại có
  `# TODO`. Hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, vẽ; khi chấm chúng bị bỏ qua. Thêm
  cell `explore` của bạn thoải mái.
- Chấm: `uv run pytest tracks/data-analyst/modules/08-visualization-chart-reading/exercises/advanced/01-revenue-drop-investigation -q` (kiểm ba hàm và `MEMO.md` đủ mục). Nội dung memo chấm theo rubric trong lời giải.

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc hai sheet (xlsx: khoảng 1 phút)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

RETAIL = ensure("online-retail-ii")
tx = pd.read_excel(RETAIL / "online_retail_II.xlsx", sheet_name="Year 2010-2011")
hist = pd.read_excel(RETAIL / "online_retail_II.xlsx", sheet_name="Year 2009-2010")
print(tx.shape, tx.InvoiceDate.min(), "→", tx.InvoiceDate.max())
print(hist.shape, hist.InvoiceDate.min(), "→", hist.InvoiceDate.max())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Làm sạch (viết sẵn)

`clean(tx)`: bỏ hoá đơn huỷ (mã bắt đầu bằng `C`), bỏ dòng `Quantity <= 0` hoặc `Price <= 0`, thêm cột
`revenue = Quantity * Price` (GBP). Đây là "doanh thu gộp" mà dashboard đang dùng.

In [ ]:
def clean(tx: pd.DataFrame) -> pd.DataFrame:
    """Các dòng mua hợp lệ, thêm cột revenue = Quantity * Price (không sửa tx)."""
    inv = tx["Invoice"].astype(str)
    mask = ~inv.str.startswith("C") & (tx["Quantity"] > 0) & (tx["Price"] > 0)
    return tx.loc[mask].assign(revenue=tx.loc[mask, "Quantity"] * tx.loc[mask, "Price"])

## 2. Dòng mua đã bị huỷ ngay sau đó

`clean` bỏ hoá đơn huỷ nhưng **giữ nguyên đơn gốc** của nó. Nếu một đơn lớn được đặt rồi huỷ sau vài phút,
doanh thu gộp vẫn đếm nó. Trước khi tin một con số so sánh, kiểm phần doanh thu "đã bị đảo".

`reversed_lines(tx) -> pd.Series` (bool, cùng index với `tx`, tên `reversed`): `True` ở **dòng mua hợp lệ**
(theo quy tắc của `clean`, và có `Customer ID`) bị một **dòng huỷ** ghép trúng. Một dòng huỷ là dòng có
`Invoice` bắt đầu bằng `C`, `Quantity < 0` và có `Customer ID`. Dòng huỷ ghép với dòng mua khi:

- cùng `Customer ID`, cùng `StockCode` (so dạng chuỗi: đọc từ xlsx, cột này trộn `int` và `str`), và
  `Quantity` mua bằng đúng `-Quantity` của dòng huỷ;
- giờ mua **không sau** giờ huỷ (`InvoiceDate` mua `<=` `InvoiceDate` huỷ);
- nếu nhiều dòng mua cùng thoả, dòng huỷ chỉ ghép với dòng mua **muộn nhất**.

Mọi dòng khác (dòng huỷ, dòng không hợp lệ, khách không có mã) là `False`.

In [ ]:
def reversed_lines(tx: pd.DataFrame) -> pd.Series:
    """True ở dòng mua hợp lệ bị một dòng huỷ ghép trúng (cùng Customer ID, StockCode, |Quantity|, huỷ không trước lúc mua)."""
    # TODO: tách dòng mua hợp lệ có Customer ID và dòng huỷ có Customer ID; StockCode đổi sang str; merge theo khoá;
    #       giữ cặp có giờ mua <= giờ huỷ; mỗi dòng huỷ chỉ lấy dòng mua muộn nhất
    raise NotImplementedError

## 3. Khách mới, khách cũ, khách vãng lai

`customer_type(tx, history) -> pd.DataFrame`: trả `clean(tx)` (cùng index, cùng cột) thêm cột
`customer_type`:

- `"guest"` khi `Customer ID` trống;
- `"new"` khi lần mua hợp lệ **đầu tiên** của khách (tính trên `clean(history)` và `clean(tx)` gộp lại) rơi
  vào **cùng tháng lịch** với dòng đang xét;
- `"returning"` trong các trường hợp còn lại.

Hoá đơn huỷ hay dòng không hợp lệ trong `history` không làm khách thành "đã từng mua".

In [ ]:
def customer_type(tx: pd.DataFrame, history: pd.DataFrame) -> pd.DataFrame:
    """clean(tx) thêm cột customer_type: "new" / "returning" / "guest" (xem quy tắc ở trên)."""
    # TODO: ngày mua hợp lệ đầu tiên của mỗi khách trên clean(history) và clean(tx);
    #       so tháng của ngày đầu tiên với tháng của từng dòng
    raise NotImplementedError

## 4. Cầu doanh thu (revenue bridge) giữa hai kỳ

`revenue_bridge(rows, a, b, by) -> pd.DataFrame`: `rows` là bảng đã làm sạch (có `InvoiceDate`, `revenue`
và cột `by`); `a`, `b` là hai kỳ dạng `(start, end)`, **`end` không tính** (`start <= InvoiceDate < end`).
Kết quả có index là các giá trị của `by` (tên index là `by`, gồm mọi giá trị có mặt ở ít nhất một kỳ) và
các cột theo đúng thứ tự:

- `a`, `b`: doanh thu của nhóm trong kỳ a, kỳ b (không có thì `0`);
- `diff = b - a`;
- `share_of_diff = diff / tổng diff`: nhóm này giải thích bao nhiêu phần của thay đổi tổng.

Sắp xếp theo `|diff|` giảm dần: dòng đầu là nhóm làm tổng thay đổi nhiều nhất (cả tăng lẫn giảm).

In [ ]:
def revenue_bridge(rows: pd.DataFrame, a: tuple, b: tuple, by: str) -> pd.DataFrame:
    """Doanh thu kỳ a và kỳ b theo `by`, cột a, b, diff, share_of_diff; sắp |diff| giảm dần."""
    # TODO: lọc [start, end) cho từng kỳ, groupby(by).revenue.sum(), ghép hai kỳ, điền 0
    raise NotImplementedError

## 5. Điều tra

Các cell dưới là điểm xuất phát, không phải đáp án. Với mỗi bước, ghi một câu **quan sát** (chart cho thấy
gì) tách với **kết luận** (bạn suy ra gì, còn cần bằng chứng gì).

1. **Chart của stakeholder**: vẽ lại để thấy cái họ thấy.
2. **Độ phủ**: tháng 12/2011 có bao nhiêu ngày giao dịch, ngày cuối có trọn ngày không?
3. **So cùng số ngày**: 1-9/12/2011 so với 1-9/11/2011 (kỳ trước) và 1-9/12/2010 (cùng kỳ năm trước). Cái
   nào trừ được mùa vụ?
4. **Ngoại lệ**: con số 1-9/12/2011 có bị vài hoá đơn quyết định không? Bao nhiêu trong đó đã bị huỷ?
5. **Tách thay đổi** (sau khi bỏ dòng đã bị huỷ): theo quốc gia, khách mới/cũ/vãng lai, sản phẩm. Thay đổi
   tập trung hay lan rộng? Khi xếp theo sản phẩm, để ý các mã không phải hàng hoá (`POST`, `DOT`,
   `AMAZONFEE`, `M`, ...).

In [ ]:
# explore: 1. chart stakeholder đang nhìn - doanh thu gộp theo tháng
rows = clean(tx)
monthly = rows.set_index("InvoiceDate")["revenue"].resample("MS").sum()
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(monthly.index, monthly / 1e6, marker="o", color="#4c78a8")
ax.set_ylim(0, None)
ax.set_ylabel("triệu GBP")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Doanh thu theo tháng (chart trên dashboard)", loc="left")
plt.show()
print(f"MoM tháng 12/2011 (ngây thơ): {monthly.iloc[-1] / monthly.iloc[-2] - 1:+.1%}")

In [ ]:
# explore: 2. độ phủ - ngày có giao dịch tháng 11-12/2011 và giờ cuối cùng có dữ liệu
daily = rows.set_index("InvoiceDate")["revenue"].resample("D").sum()
recent = daily["2011-11-01":]
print("giao dịch cuối:", rows.InvoiceDate.max())
print("số ngày có giao dịch tháng 11/2011:", (daily["2011-11"] > 0).sum(), "| tháng 12/2011:", (daily["2011-12"] > 0).sum())
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(recent.index, recent / 1e3, color="#9aa5b1")
ax.set_ylabel("nghìn GBP / ngày")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Doanh thu mỗi ngày, 1/11 - 9/12/2011", loc="left")
plt.show()

In [ ]:
# explore: 3. so cùng số ngày - doanh thu cộng dồn theo ngày trong tháng, ngày 1-9
WINDOWS = {"1-9/12/2011": ("2011-12-01", "2011-12-10"),
           "1-9/11/2011": ("2011-11-01", "2011-11-10"),
           "1-9/12/2010": ("2010-12-01", "2010-12-10")}


def cumulative(frame, start, end):
    part = frame[(frame.InvoiceDate >= start) & (frame.InvoiceDate < end)]
    return part.groupby(part.InvoiceDate.dt.day)["revenue"].sum().reindex(range(1, 10), fill_value=0).cumsum()


fig, ax = plt.subplots(figsize=(8, 3.5))
for (label, (start, end)), color in zip(WINDOWS.items(), ["#4c78a8", "#9aa5b1", "#f58518"]):
    cum = cumulative(rows, start, end)
    ax.plot(cum.index, cum / 1e3, marker="o", color=color, label=label)
ax.set_xlabel("ngày trong tháng")
ax.set_ylabel("nghìn GBP cộng dồn")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()
print({label: round(cumulative(rows, *w).iloc[-1]) for label, w in WINDOWS.items()})

In [ ]:
# explore: 4. ngoại lệ - hoá đơn lớn nhất 1-9/12/2011 và phần doanh thu đã bị huỷ
flag = reversed_lines(tx)
rows = rows.assign(reversed=flag.loc[rows.index])
dec = rows[(rows.InvoiceDate >= "2011-12-01") & (rows.InvoiceDate < "2011-12-10")]
print(dec.groupby("Invoice").agg(revenue=("revenue", "sum"), reversed=("reversed", "any"),
                                 customer=("Customer ID", "first"), at=("InvoiceDate", "min"))
      .sort_values("revenue", ascending=False).head(5))
for label, (start, end) in WINDOWS.items():
    part = rows[(rows.InvoiceDate >= start) & (rows.InvoiceDate < end)]
    print(f"{label}: gộp {part.revenue.sum():>10,.0f}  đã bị huỷ {part.revenue[part.reversed].sum():>9,.0f}"
          f"  còn lại {part.revenue[~part.reversed].sum():>10,.0f}")

In [ ]:
# explore: 5. tách thay đổi cùng kỳ năm trước theo quốc gia, loại khách, sản phẩm (đã bỏ dòng bị huỷ)
typed = customer_type(tx, hist)
kept = typed[~flag.loc[typed.index]]
A, B = WINDOWS["1-9/12/2010"], WINDOWS["1-9/12/2011"]
for by in ["Country", "customer_type", "StockCode"]:
    bridge = revenue_bridge(kept, A, B, by)
    print(f"--- {by}: tổng diff {bridge['diff'].sum():,.0f}")
    print(bridge.head(6).round(3).to_string())

In [ ]:
# explore: chart của bạn cho memo (mỗi chart: tiêu đề là kết luận, có con số, có annotation khi cần)

## 6. Viết `MEMO.md`

Mở `starter/MEMO.md`, điền theo khung có sẵn: headline là kết luận có con số, trả lời ngắn, **3 chart** (mỗi
chart ghi headline, loại chart, cell nào trong notebook), kết luận, việc cần làm (ai, làm gì, khi nào xem lại)
và giới hạn. 200-600 từ, xoá mọi dòng `TODO`. Người đọc là Giám đốc kinh doanh, không phải DA: không kể quá
trình làm, không dùng tên cột.